# F8 : Battre la marche aléatoire

**Nom et prénom :** MALAK SABRI   **Code Apogée :** 24010191   **Groupe :** FIN 3   **Filière :** FINANCE   **Date :** 17/11/2026

**Dépôt GitHub :**https://github.com/malaksabri65/24010191-SABRI-FIN3.git

## 1. Sujet et problématique
- **Intérêt du sujet** : si les rendements d'un indice boursier suivent une marche aléatoire, aucune règle fondée sur les cours passés ne peut battre le hasard (hypothèse d'efficience faible). Un gérant ou un étudiant en finance veut savoir si les rendements et la volatilité récents permettent de prévoir le signe du lendemain, et décider s'il vaut la peine de bâtir une stratégie dessus.
- **Problématique** : un modèle de classification utilisant uniquement les rendements passés du S&P 500 prévoit-il le signe du rendement du jour suivant mieux que la référence naïve, sur une période de test postérieure à l'apprentissage ?
- **Objectif mesurable (fixé avant de voir le test)** : métrique principale = accuracy sur le test (20 % des données les plus récentes). Réussite si l'accuracy dépasse d'au moins 2 points la meilleure des deux références (classe majoritaire de l'apprentissage ; signe de la veille) **et** si un test binomial unilatéral contre cette référence est significatif à 5 %. Métriques secondaires : F1 et AUC.
- **Nature du problème** : classification binaire sur série temporelle (cible = 1 si le rendement du jour suivant est strictement positif, 0 sinon) : découpage chronologique obligatoire.
- **Algorithme principal** : n° 8, régression logistique. Comparaison facultative : n° 12, forêt aléatoire.

## Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date
from IPython.display import Markdown, display
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

CHEMIN = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=SP500"
DATE_EXTRACTION = date.today().strftime("%d/%m/%Y")

def dire(texte):
    display(Markdown(texte))

## 2. Jeu de données

In [ ]:
try:
    brut = pd.read_csv(CHEMIN)
except Exception as e:
    print("Téléchargement automatique impossible :", e)
    print("Importez le fichier CSV téléchargé sur FRED (série SP500) :")
    from google.colab import files
    up = files.upload()
    brut = pd.read_csv(list(up.keys())[0])

print("Dimensions brutes :", brut.shape)
print(brut.dtypes)
print(brut.head())

In [ ]:
col_date = [c for c in brut.columns if c.lower() in ("observation_date", "date")][0]
col_val = [c for c in brut.columns if c != col_date][0]
df = brut.rename(columns={col_date: "date", col_val: "prix"})
df["date"] = pd.to_datetime(df["date"])
df["prix"] = pd.to_numeric(df["prix"], errors="coerce")
n_manq = int(df["prix"].isna().sum())
pct_manq = df["prix"].isna().mean() * 100
df = df.set_index("date").sort_index()
n_dup = int(df.index.duplicated().sum())
df = df.dropna()
print(f"Valeurs manquantes : {n_manq} ({pct_manq:.2f} %) | doublons de dates : {n_dup}")
print("Observations :", len(df), "| période :", df.index.min().date(), "->", df.index.max().date())

In [ ]:
dire(f"""
- **Nom et source** : S&P 500 (cours de clôture quotidiens), Federal Reserve Bank of St. Louis (FRED), série `SP500`. Lien : https://fred.stlouisfed.org/series/SP500
- **Extraction** : {DATE_EXTRACTION}.
- **Licence** : données S&P Dow Jones Indices diffusées par FRED pour usage d'information ; usage ici strictement pédagogique (à confirmer sur la page « Notes » de la série).
- **Taille et fréquence** : {len(df)} observations quotidiennes (jours de bourse), du {df.index.min().date()} au {df.index.max().date()}.
- **Unité d'observation** : un jour de bourse.
- **Adéquation** : la question porte sur la prévisibilité du signe du rendement quotidien ; la série de clôtures suffit à construire la cible et les variables passées.
- **Limites** : environ {round((df.index.max()-df.index.min()).days/365.25)} ans de données seulement ; une seule série ; contient le krach de mars 2020 (régime de forte volatilité) ; prix sans dividendes ; aucun coût de transaction ; peu d'observations pour détecter un faible signal.
""")

### Dictionnaire des variables

| Variable | Description | Type | Unité | Rôle | Manquants |
| --- | --- | --- | --- | --- | --- |
| prix | Cours de clôture du S&P 500 | numérique | points d'indice | brute | jours sans cotation retirés (section 3) |
| ret | Rendement logarithmique quotidien 100 × ln(P_t / P_{t-1}) | numérique | % | base des variables | 1re ligne retirée |
| ret_lag1 à ret_lag5 | ret décalé de 1 à 5 jours | numérique | % | explicative | aucun après nettoyage |
| moy5, moy20 | moyenne glissante de ret sur 5 / 20 jours se terminant en t | numérique | % | explicative | aucun après nettoyage |
| vol5, vol20 | écart-type glissant de ret sur 5 / 20 jours se terminant en t | numérique | % | explicative | aucun après nettoyage |
| y | 1 si le rendement de t+1 est > 0, sinon 0 | binaire | sans | **cible** | dernière ligne retirée |

## 3. Construction des variables

In [ ]:
df["ret"] = 100 * np.log(df["prix"] / df["prix"].shift(1))
for k in range(1, 6):
    df[f"ret_lag{k}"] = df["ret"].shift(k)
df["moy5"] = df["ret"].rolling(5).mean()
df["moy20"] = df["ret"].rolling(20).mean()
df["vol5"] = df["ret"].rolling(5).std()
df["vol20"] = df["ret"].rolling(20).std()
df["y"] = (df["ret"].shift(-1) > 0).astype(int)
df = df.iloc[:-1]      # dernière ligne : lendemain inconnu
df = df.dropna()       # début de série (fenêtres glissantes incomplètes)

X_cols = [f"ret_lag{k}" for k in range(1, 6)] + ["moy5", "moy20", "vol5", "vol20"]
print("Observations utilisables :", len(df))
print("Part de jours haussiers (y=1) :", round(df["y"].mean() * 100, 2), "%")
df.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
df["prix"].plot(ax=ax[0], title="S&P 500 - cours de clôture")
df["ret"].plot(ax=ax[1], title="Rendement logarithmique quotidien (%)", lw=0.6)
plt.tight_layout(); plt.show()

## 4. Découpage chronologique (80 % apprentissage / 20 % test)

In [ ]:
coupure = int(len(df) * 0.8)
train, test = df.iloc[:coupure], df.iloc[coupure:]
X_tr, y_tr = train[X_cols], train["y"]
X_te, y_te = test[X_cols], test["y"]
print(f"Apprentissage : {len(train)} jours ({train.index.min().date()} -> {train.index.max().date()})")
print(f"Test          : {len(test)} jours ({test.index.min().date()} -> {test.index.max().date()})")

## 5. Références naïves

In [ ]:
classe_maj = int(y_tr.mean() >= 0.5)
acc_maj = accuracy_score(y_te, np.full(len(y_te), classe_maj))
acc_veille = accuracy_score(y_te, (test["ret"] > 0).astype(int))   # signe d'aujourd'hui = prévision de demain
meilleure_ref = max(acc_maj, acc_veille)
nom_ref = "classe majoritaire" if acc_maj >= acc_veille else "signe de la veille"
print(f"Classe majoritaire (classe {classe_maj}) : {acc_maj*100:.2f} %")
print(f"Signe de la veille                : {acc_veille*100:.2f} %")
print(f"Meilleure référence : {nom_ref} ({meilleure_ref*100:.2f} %)")

## 6. Modèle principal : régression logistique

In [ ]:
modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modele.fit(X_tr, y_tr)
pred = modele.predict(X_te)
proba = modele.predict_proba(X_te)[:, 1]

acc = accuracy_score(y_te, pred)
f1 = f1_score(y_te, pred)
auc = roc_auc_score(y_te, proba)

n_test = len(y_te)
n_ok = int((pred == y_te.values).sum())
p_val = binomtest(n_ok, n_test, p=meilleure_ref, alternative="greater").pvalue
critere = (acc >= meilleure_ref + 0.02) and (p_val < 0.05)

print(f"Accuracy : {acc*100:.2f} %  |  F1 : {f1:.3f}  |  AUC : {auc:.3f}")
print(f"Écart vs meilleure référence : {(acc-meilleure_ref)*100:+.2f} points")
print(f"p-valeur du test binomial unilatéral : {p_val:.4f}")
print("CRITÈRE :", "ATTEINT" if critere else "NON ATTEINT")

## 7. Comparaison facultative : forêt aléatoire

In [ ]:
rf = RandomForestClassifier(n_estimators=300, max_depth=5, random_state=42)
rf.fit(X_tr, y_tr)
acc_rf = accuracy_score(y_te, rf.predict(X_te))
auc_rf = roc_auc_score(y_te, rf.predict_proba(X_te)[:, 1])
print(f"Forêt aléatoire : accuracy {acc_rf*100:.2f} % | AUC {auc_rf:.3f}")

## 8. Résultats et conclusion

In [ ]:
noms = ["Classe majoritaire", "Signe de la veille", "Rég. logistique", "Forêt aléatoire"]
vals = [acc_maj*100, acc_veille*100, acc*100, acc_rf*100]
plt.figure(figsize=(7, 4))
plt.bar(noms, vals)
plt.axhline(meilleure_ref*100 + 2, ls="--", label="Seuil de réussite (+2 pts)")
plt.ylim(min(vals) - 5, max(vals) + 5)
plt.ylabel("Accuracy sur le test (%)"); plt.title("Modèle vs références")
plt.xticks(rotation=15); plt.legend(); plt.tight_layout(); plt.show()

print("=" * 55)
print("RÉSULTATS POUR LA VIDÉO (copiez ce bloc)")
print("=" * 55)
print(f"Valeurs manquantes : {n_manq} | doublons de dates : {n_dup}")
print(f"Période : {df.index.min().date()} -> {df.index.max().date()}")
print(f"Observations utilisables : {len(df)}")
print(f"Apprentissage : {len(train)} | Test : {n_test}")
print(f"Accuracy classe majoritaire : {acc_maj*100:.2f} %")
print(f"Accuracy signe de la veille : {acc_veille*100:.2f} %")
print(f"Meilleure référence ({nom_ref}) : {meilleure_ref*100:.2f} %")
print(f"Rég. logistique - accuracy : {acc*100:.2f} % | F1 : {f1:.3f} | AUC : {auc:.3f}")
print(f"Écart vs meilleure référence : {(acc-meilleure_ref)*100:+.2f} points")
print(f"p-valeur binomiale : {p_val:.4f}")
print(f"Critère : {'ATTEINT' if critere else 'NON ATTEINT'}")
print(f"Forêt aléatoire - accuracy : {acc_rf*100:.2f} % | AUC : {auc_rf:.3f}")
print("=" * 55)